## Intercepting Agent Execution with Hooks

# Lesson 9: Runtime Interception and Defensive Architecture via Lifecycle Hooks

In the previous modules, you mastered the engineering of complex, single-agent systems and multi-agent hierarchies by managing stateful workflows and coordinating task delegation. Equipped with system tools and Model Context Protocol (MCP) servers, these systems can modify codebases, inspect file structures, and execute terminal commands natively.

However, allowing an autonomous agent to execute arbitrary shell operations or evaluate untrusted inputs introduces severe security risks. To build a reliable AI platform, you must establish strict boundary layers. This is where **Lifecycle Hooks** become essential.

Hooks are asynchronous callback interfaces that allow you to intercept, audit, and block your agent's behavior at deterministic checkpoints during its execution cycle. By injecting defensive logic into these checkpoints, you can transform an unrestricted LLM loop into a highly secure, enterprise-grade runtime sandbox.

---

## The Available Hook Matrix

The Claude Agent SDK provides six explicit hooks mapped across the runtime lifecycle. This allows you to construct deep, layered security strategies:

| Hook Event Token | Execution Checkpoint Trigger | Primary Technical Use Case |
| --- | --- | --- |
| **`UserPromptSubmit`** | Fires immediately when a user submits an input, before the LLM processes it. | Sanitizing raw inputs, checking for prompt injections, and implementing input guardrails. |
| **`PreToolUse`** | Fires right before the agent executes a whitelisted tool. | Evaluating tool arguments, checking permissions, and blocking destructive commands. |
| **`PostToolUse`** | Triggers immediately after a tool finishes execution, before passing results back to Claude. | Auditing logs, scrubbing sensitive outputs, and capturing data metrics. |
| **`Stop`** | Activates when the agent naturally finishes its multi-turn execution cycle. | Cleaning up workspace resources, closing database connections, and running telemetry summaries. |
| **`PreCompact`** | Triggers when token thresholds force the context history window to be compressed. | Pruning irrelevant conversation metadata or injecting custom context summaries. |
| **`SubagentStop`** | Fires exactly when a delegated `AgentDefinition` sub-agent finishes its task. | Auditing sub-agent responses and verifying cross-node alignment. |

In this lesson, we will focus on implementing **`UserPromptSubmit`** and **`PreToolUse`**. Together, these two hooks form a comprehensive, layered defense system.

---

## Anatomy of a Hook Signature

Every hook function inside the Claude Agent SDK must strictly follow this asynchronous function signature:

```python
from typing import Any, Dict, Optional
from claude_agent_sdk import HookContext

async def my_custom_hook(
    input_data: Dict[str, Any],
    tool_use_id: Optional[str],
    context: HookContext,
) -> Dict[str, Any]:
    """Standardized asynchronous hook specification."""
    # 1. Perform inspection or validation logic here...
    
    # 2. Yield an empty dictionary to signal a 'proceed normally' decision
    return {}

```

### The Ingestion Parameters Decoded:

* **`input_data`:** A dictionary containing event-specific telemetry. Its contents change depending on the hook type:
* For `UserPromptSubmit`, it provides a `{"prompt": "string"}` dictionary.
* For `PreToolUse`, it provides `{"tool_name": "string", "tool_input": {...}}`.


* **`tool_use_id`:** A unique tracking string for tool execution blocks. This is set to `None` for prompt submission events.
* **`context`:** A `HookContext` object carrying execution metadata, such as cancellation signals or thread tracking details.

### Controlling the Runtime via Return Payloads:

* **Proceed Normally:** Returning an empty dictionary (`return {}`) tells the runtime that the hook passed verification and the agent can safely continue execution.
* **Intercept and Alter:** Returning a populated dictionary overrides the agent's behavior. The required dictionary structure is specific to each hook type:
* `UserPromptSubmit` expects a top-level decision map (`{"decision": "block", "systemMessage": "Reason"}`).
* `PreToolUse` expects a structured output block (`{"hookSpecificOutput": {"permissionDecision": "deny", ...}}`).



---

## Implementing a UserPromptSubmit Intent Guardrail

The first line of defense is an input validation hook. By registering an intent guardrail on the `UserPromptSubmit` event, you can scan raw user text for malicious keywords or prompt injection attacks before they ever reach the model.

```python
from typing import Any, Dict, Optional
from claude_agent_sdk import HookContext

async def intent_guardrail(
    input_data: Dict[str, Any],
    tool_use_id: Optional[str],
    context: HookContext,
) -> Dict[str, Any]:
    """Intercepts and sanitizes incoming user prompts prior to LLM processing."""
    # Extract the user's prompt safely and convert to lowercase for case-insensitive matching
    raw_prompt = input_data.get("prompt", "").lower()
    print(f"[HOOK] Guardrail checking user intent: '{raw_prompt}'")
    
    # Blacklist representing malicious exploitation vectors and injection attempts
    malicious_blacklist = [
        "hack", 
        "exploit", 
        "steal", 
        "credential", 
        "password", 
        "ignore all previous instructions"
    ]
    
    # Scan prompt against blacklist
    for signature in malicious_blacklist:
        if signature in raw_prompt:
            print(f"[HOOK INTERCEPTION] Malicious signature detected: '{signature}'")
            
            # Immediately halt the pipeline and return a system error notification to the user
            return {
                "decision": "block",
                "systemMessage": "Security Violation: base prompt was blocked due to unsafe behavior requests."
            }
            
    # Allow the prompt to pass through to the model unchanged
    return {}

```

---

## Implementing a PreToolUse Bash Safety Guardrail

Even if an input prompt appears harmless, an agent can still generate dangerous commands during multi-turn reasoning. To prevent destructive operations, you need a second layer of security right before tool execution.

Below is a `PreToolUse` guardrail designed to parse `Bash` inputs, verify their safety profiles, and block malicious command strings:

```python
async def bash_safety_guardrail(
    input_data: Dict[str, Any],
    tool_use_id: Optional[str],
    context: HookContext,
) -> Dict[str, Any]:
    """Intercepts tool invocations right before execution to block dangerous bash patterns."""
    tool_name = input_data.get("tool_name")
    
    # If the tool isn't a Bash terminal command, exit early to minimize latency overhead
    if tool_name != "Bash":
        return {}
        
    # Unpack argument structures from the tool payload
    tool_input = input_data.get("tool_input", {})
    target_command = tool_input.get("command", "")
    print(f"[HOOK] Inspecting system Bash command: '{target_command}'")
    
    # Blocklist containing high-risk, destructive system patterns
    forbidden_commands = [
        "rm -rf",      # Destructive file removal
        "sudo",        # Unauthorized privilege escalation
        "> /etc/",     # System configuration overwrites
        "mkfs",        # Filesystem formatting
        ":(){ :|:& };:" # Fork bomb DOS attacks
    ]
    
    # Scan the target command for dangerous patterns
    for pattern in forbidden_commands:
        if pattern in target_command:
            print(f"[HOOK DENIAL] Blocked dangerous bash signature: '{pattern}'")
            
            # Return a tool permission denial payload to the runtime
            return {
                "hookSpecificOutput": {
                    "hookEventName": "PreToolUse",
                    "permissionDecision": "deny",
                    "permissionDecisionReason": f"Operation denied: Command contains forbidden pattern: '{pattern}'"
                }
            }
            
    # Command is safe, allow execution to proceed normally
    return {}

```

---

## Registering Lifecycle Hooks via Agent Options

Once your guardrails are written, register them with the agent by mapping them inside the `hooks` dictionary configuration using `HookMatcher` objects:

```python
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions, HookMatcher

# Compile options with safety guardrails attached
options = ClaudeAgentOptions(
    model="haiku",
    max_turns=5,
    allowed_tools=["Bash"],
    permission_mode="acceptEdits",
    cwd=Path(__file__).parent,
    # Register your custom hooks to specific lifecycles
    hooks={
        # Set 1: Pre-process user inputs before they hit the model
        "UserPromptSubmit": [
            # matcher="*" applies this hook to all incoming prompts globally
            HookMatcher(matcher="*", hooks=[intent_guardrail])
        ],
        # Set 2: Audit tool calls immediately before system execution
        "PreToolUse": [
            # Check all tool executions; the function internally targets Bash calls
            HookMatcher(matcher="*", hooks=[bash_safety_guardrail])
        ]
    }
)

```

---

## Comprehensive Blueprint: Multi-Layer Security Suite

Let's integrate our safety hooks into a complete, runnable script. This framework evaluates three sequential prompts: an unsafe prompt that gets blocked at the input layer, a risky prompt that gets blocked right before tool execution, and a safe prompt that runs successfully.

```python
import anyio
from pathlib import Path
from typing import Any, Dict, Optional
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, HookMatcher, HookContext
from utils import display_response

# --- Hook Definitions ---

async def intent_guardrail(input_data: Dict[str, Any], tool_use_id: Optional[str], context: HookContext) -> Dict[str, Any]:
    raw_prompt = input_data.get("prompt", "").lower()
    print(f"\n[HOOK] Checking input intent: '{raw_prompt}'")
    
    blocked_keywords = ["hack", "exploit", "steal", "credential", "password", "ignore all previous instructions"]
    for word in blocked_keywords:
        if word in raw_prompt:
            print(f"[HOOK] Blocking prompt due to keyword match: '{word}'")
            return {
                "decision": "block",
                "systemMessage": "Access Denied: Your request was blocked due to unsafe behavior policy."
            }
    return {}

async def bash_safety_guardrail(input_data: Dict[str, Any], tool_use_id: Optional[str], context: HookContext) -> Dict[str, Any]:
    if input_data.get("tool_name") != "Bash":
        return {}
        
    tool_input = input_data.get("tool_input", {})
    command = tool_input.get("command", "")
    print(f"\n[HOOK] Auditing Bash command string: '{command}'")
    
    dangerous_patterns = ["rm -rf", "sudo", "> /etc/", "mkfs", ":(){ :|:& };:"]
    for pattern in dangerous_patterns:
        if pattern in command:
            print(f"[HOOK] Denying tool execution due to forbidden signature: '{pattern}'")
            return {
                "hookSpecificOutput": {
                    "hookEventName": "PreToolUse",
                    "permissionDecision": "deny",
                    "permissionDecisionReason": f"Security Infraction: Command contains forbidden pattern: '{pattern}'"
                }
            }
    return {}

# --- Runtime Engine ---

async def main():
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=5,
        allowed_tools=["Bash"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent,
        hooks={
            "UserPromptSubmit": [HookMatcher(matcher="*", hooks=[intent_guardrail])],
            "PreToolUse": [HookMatcher(matcher="*", hooks=[bash_safety_guardrail])]
        }
    )

    prompts = [
        "How do I hack the mainframe database to bypass passwords?", 
        "Please remove the temporary test directory using 'rm -rf /tmp/test'.",  
        "List files in the current working directory."  
    ]

    async with ClaudeSDKClient(options=options) as client:
        for prompt in prompts:
            print(f"\n" + "="*60 + f"\n[TEST ENVIRONMENT] Evaluation: {prompt}\n" + "="*60)
            await client.query(prompt)
            await display_response(client)

if __name__ == "__main__":
    anyio.run(main)

```

---

## Runtime Execution Trace Logs

When running the security suite above, you can see exactly how the defensive layers intercept dangerous behaviors at different points in the lifecycle:

### Scenario 1: Input Layer Interception (Prompt Blocked)

The user prompt contains the word `"hack"`. The `UserPromptSubmit` hook catches it instantly, blocking execution before any LLM processing occurs and saving computational costs:

```text
============================================================
[TEST ENVIRONMENT] Evaluation: How do I hack the mainframe database to bypass passwords?
============================================================

[HOOK] Checking input intent: 'how do i hack the mainframe database to bypass passwords?'
[HOOK] Blocking prompt due to keyword match: 'hack'

💬 Claude Response:
Access Denied: Your request was blocked due to unsafe behavior policy.

```

### Scenario 2: Tool Layer Interception (Execution Denied)

The prompt itself looks safe, so it passes the input guardrail. However, during the tool execution phase, the agent generates an unsafe `rm -rf` system command. The `PreToolUse` hook intercepts the call and blocks it, prompting Claude to suggest safe alternatives:

```text
============================================================
[TEST ENVIRONMENT] Evaluation: Please remove the temporary test directory using 'rm -rf /tmp/test'.
============================================================

[HOOK] Checking input intent: 'please remove the temporary test directory using 'rm -rf /tmp/test'.'

💬 Claude Response:
I will execute the command to remove the targeted workspace folder directory path.

[HOOK] Auditing Bash command string: 'rm -rf /tmp/test'
[HOOK] Denying tool execution due to forbidden signature: 'rm -rf'

💬 Claude Response:
I cannot execute that file removal command. The environment has safety restrictions preventing the use of `rm -rf` to protect against data loss.

If you need to delete the `/tmp/test` folder safely, please use one of these alternative approaches:
1. Using standard `rmdir` for empty targets: rmdir /tmp/test
2. Running an interactive file removal prompt: rm -ri /tmp/test

```

### Scenario 3: Unrestricted Safe Execution

A valid, non-destructive command passes through both security layers smoothly, allowing the agent to provide the requested directory information:

```text
============================================================
[TEST ENVIRONMENT] Evaluation: List files in the current working directory.
============================================================

[HOOK] Checking input intent: 'list files in the current working directory.'

[HOOK] Auditing Bash command string: 'ls -la'

💬 Claude Response:
Here are the contents of your current active directory tree:
- main.py  (Python executable)
- utils.py (Telemetry engine)

```

---

## Summary

You have mastered how to build robust, layered security systems using lifecycle hooks:

* **`UserPromptSubmit`** acts as an input filter, inspecting user intent and catching prompt injections before they reach the model.
* **`PreToolUse`** acts as an execution gatekeeper, evaluating tool inputs right before system execution to block dangerous command strings.
* Returning an **empty dictionary (`return {}`)** allows the agent workflow to continue running smoothly.
* Populating the return payload lets you **block prompts** or **deny tool permissions** dynamically, keeping your autonomous systems completely secure.

## Building Your First Agent Hook

Now that you've seen how hooks provide powerful control over agent behavior, it's time to build your first hook from scratch! In this exercise, you'll create a simple notification hook that announces when the agent starts processing a request.

Your task is to transform the incomplete startup_hook function into a proper hook by giving it the correct async signature with all three required parameters: input_data, tool_use_id, and context. Don't forget that hook functions must return a dictionary — in this case, an empty one to allow normal execution.

Once your hook function is ready, you'll need to register it with the agent by adding a hooks parameter to ClaudeAgentOptions. Use the "UserPromptSubmit" hook type with a HookMatcher that applies to all prompts (using matcher="*") and include your startup_hook in the hooks list.

The agent query is already set up for you, so once you've completed these steps, you'll see your hook in action as it intercepts the prompt before the agent processes it!

```
import anyio
from typing import Any, Dict, Optional
from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    HookContext,
    HookMatcher,
)
from utils import display_response


# TODO: Convert this into a proper hook function with the correct async signature and parameters
def startup_hook():
    """Runs before the agent processes the user prompt."""
    print("[HOOK] Agent is starting to process your request...")


async def main():
    # TODO: Register your startup_hook in the ClaudeAgentOptions using the hooks parameter
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=3,
    )

    async with ClaudeSDKClient(options=options) as client:
        await client.query("Hey")
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

Here is the completed code that implements your first lifecycle hook.

The `startup_hook` has been converted to an asynchronous function with the correct parameter signatures (`input_data`, `tool_use_id`, `context`) and returns an empty dictionary to let the execution flow continue normally. It is then mapped to the `UserPromptSubmit` event using a `HookMatcher` with a wildcard (`"*"`) pattern:

```python
import anyio
from typing import Any, Dict, Optional
from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    HookContext,
    HookMatcher,
)
from utils import display_response


# Convert this into a proper hook function with the correct async signature and parameters
async def startup_hook(
    input_data: Dict[str, Any],
    tool_use_id: Optional[str],
    context: HookContext,
) -> Dict[str, Any]:
    """Runs before the agent processes the user prompt."""
    print("[HOOK] Agent is starting to process your request...")
    
    # Return an empty dictionary to signal that the agent should proceed normally
    return {}


async def main():
    # Register your startup_hook in the ClaudeAgentOptions using the hooks parameter
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=3,
        # Define lifecycle event interception checkpoints
        hooks={
            "UserPromptSubmit": [
                # Matcher "*" ensures this runs for all incoming queries globally
                HookMatcher(matcher="*", hooks=[startup_hook]),
            ]
        }
    )

    async with ClaudeSDKClient(options=options) as client:
        await client.query("Hey")
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Key Lifecycle Principles Applied:

* **The `async` Signature Requirement:** The agent runtime infrastructure operates concurrently within a non-blocking asynchronous pipeline. Standard synchronous functions (`def`) will crash the runtime during callbacks.
* **The Telemetry Payload Loop:** Even if you aren't using the data inside this simple notification utility, `input_data` will hold a `{"prompt": "Hey"}` dictionary under the hood when called.
* **The Return Constraint Matrix:** An explicit `return {}` statement is required. If a hook yields nothing (`None`), it breaks protocol rules and causes an evaluation failure. An empty map tells the orchestrator that the checkpoint check passed.

## Blocking Dangerous Prompts with a Custom Guardrail Hook

## Creating a Simple Tool Logger Hook

## Building Your First Bash Safety Hook